In [ ]:
!pip install openai-whisper librosa scikit-learn pyannote.audio torch torchaudio pydub requests aiohttp transformers sentence-transformers google-generativeai

In [ ]:
# Enhanced Interview Audio Analyzer with Multiple LLM Providers and Advanced AI Detection
# Run this in Google Colab to analyze interview recordings with comprehensive AI evaluation

import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
import librosa
import whisper
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
import re
from datetime import timedelta
import warnings
import requests
import json
import time
from typing import Dict, List, Optional, Tuple
import hashlib
import asyncio
import aiohttp
warnings.filterwarnings('ignore')

# Install required packages
def install_packages():
    """Install required packages in Colab"""
    packages = [
        "openai-whisper",
        "librosa",
        "scikit-learn",
        "pyannote.audio",
        "torch",
        "torchaudio",
        "pydub",
        "requests",
        "aiohttp",
        "transformers",
        "sentence-transformers",
        "google-generativeai"
    ]

    for package in packages:
        try:
            subprocess.check_call([f"pip install {package}"], shell=True)
            print(f"✓ {package} installed successfully")
        except subprocess.CalledProcessError:
            print(f"✗ Failed to install {package}")

import requests
import re
import time
import json
import asyncio
import aiohttp
from typing import Optional, Dict, Tuple, List

class MultiLLMEvaluator:
    """Handles evaluation using multiple free LLM providers for robust assessment"""

    def __init__(self):
        """Initialize the evaluator with API keys collected during runtime"""
        self.api_keys = {}
        self.endpoints = {
            'huggingface': {
                'base_url': 'https://api-inference.huggingface.co/models/',
                'models': [
                    'mistralai/Mistral-7B-Instruct-v0.2',
                    'microsoft/DialoGPT-medium',
                    'meta-llama/Llama-2-7b-chat-hf',
                    'google/flan-t5-large',
                    'HuggingFaceH4/zephyr-7b-beta'
                ]
            },
            'groq': {
                'base_url': 'https://api.groq.com/openai/v1/chat/completions',
                'models': ['llama3-8b-8192', 'mixtral-8x7b-32768']
            },
            'google': {
                'base_url': 'https://generativelanguage.googleapis.com/v1beta/models/',
                'models': ['gemini-pro', 'gemini-1.5-flash']
            }
        }
        self.ai_detection_services = [
            'gptzero',
            'google_ai_detection'
        ]

    def collect_api_keys(self):
        """Collect API keys from user during execution"""
        print("\n🔑 API Key Collection for Enhanced Features")
        print("=" * 50)
        print("Please provide API keys for the services you want to use.")
        print("You can skip any service by pressing Enter (will use fallback methods).\n")

        # Hugging Face (Free tier available)
        hf_key = input("🤗 Hugging Face API Key (free at https://huggingface.co/settings/tokens): ").strip()
        if hf_key:
            self.api_keys['huggingface'] = hf_key
            print("✓ Hugging Face API key added")

        # Groq (Free tier available)
        groq_key = input("🚀 Groq API Key (free at https://console.groq.com/): ").strip()
        if groq_key:
            self.api_keys['groq'] = groq_key
            print("✓ Groq API key added")

        # Google Gemini API (Free tier available)
        google_key = input("🔍 Google Gemini API Key (free at https://aistudio.google.com/app/apikey): ").strip()
        if google_key:
            self.api_keys['google'] = google_key
            print("✓ Google Gemini API key added")

        # GPTZero for AI detection
        gptzero_key = input("🔍 GPTZero API Key (for AI detection, free tier at https://gptzero.me/): ").strip()
        if gptzero_key:
            self.api_keys['gptzero'] = gptzero_key
            print("✓ GPTZero API key added")

        print(f"\n✅ Collected {len(self.api_keys)} API keys for enhanced analysis")

    def score_answer_with_llm(self, question: str, answer: str, context: str = "") -> Dict:
        """Score answer using multiple LLM providers for consensus"""

        evaluation_prompt = f"""You are an expert interview evaluator. Analyze this interview response and provide a detailed evaluation.

INTERVIEW QUESTION: {question}

CANDIDATE'S ANSWER: {answer}

JOB CONTEXT: {context}

Please provide your evaluation in this EXACT format:
SCORE: [number from 1-10]
REASONING: [2-3 sentences explaining your score]
STRENGTHS: [bullet points of strengths]
WEAKNESSES: [bullet points of areas for improvement]
RECOMMENDATION: [brief hiring recommendation]"""

        results = []

        # Try Hugging Face models
        if 'huggingface' in self.api_keys:
            hf_result = self._evaluate_with_huggingface(evaluation_prompt)
            if hf_result:
                results.append(hf_result)

        # Try Groq
        if 'groq' in self.api_keys:
            groq_result = self._evaluate_with_groq(evaluation_prompt)
            if groq_result:
                results.append(groq_result)

        # Try Google Gemini
        if 'google' in self.api_keys:
            google_result = self._evaluate_with_google(evaluation_prompt)
            if google_result:
                results.append(google_result)

        if results:
            return self._aggregate_llm_results(results)
        else:
            # Ultimate fallback - basic heuristic scoring
            return self._basic_heuristic_evaluation(question, answer)

    def _evaluate_with_huggingface(self, prompt: str) -> Optional[Dict]:
        """Evaluate using Hugging Face Inference API"""
        try:
            headers = {"Authorization": f"Bearer {self.api_keys['huggingface']}"}

            for model in self.endpoints['huggingface']['models']:
                try:
                    response = requests.post(
                        f"{self.endpoints['huggingface']['base_url']}{model}",
                        headers=headers,
                        json={
                            "inputs": prompt,
                            "parameters": {
                                "max_new_tokens": 300,
                                "temperature": 0.3,
                                "return_full_text": False
                            }
                        },
                        timeout=30
                    )

                    if response.status_code == 200:
                        result = response.json()
                        if isinstance(result, list) and result:
                            text = result[0].get('generated_text', '')
                            return self._parse_llm_response(text, 'huggingface')

                except Exception as e:
                    continue

        except Exception as e:
            print(f"Hugging Face evaluation failed: {e}")
        return None

    def _evaluate_with_groq(self, prompt: str) -> Optional[Dict]:
        """Evaluate using Groq API"""
        try:
            headers = {
                "Authorization": f"Bearer {self.api_keys['groq']}",
                "Content-Type": "application/json"
            }

            payload = {
                "messages": [{"role": "user", "content": prompt}],
                "model": "llama3-8b-8192",
                "temperature": 0.3,
                "max_tokens": 300
            }

            response = requests.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                text = result['choices'][0]['message']['content']
                return self._parse_llm_response(text, 'groq')

        except Exception as e:
            print(f"Groq evaluation failed: {e}")
        return None

    def _evaluate_with_google(self, prompt: str) -> Optional[Dict]:
        """Evaluate using Google Gemini API"""
        try:
            import google.generativeai as genai

            genai.configure(api_key=self.api_keys['google'])

            for model_name in self.endpoints['google']['models']:
                try:
                    model = genai.GenerativeModel(model_name)
                    response = model.generate_content(
                        prompt,
                        generation_config=genai.types.GenerationConfig(
                            temperature=0.3,
                            max_output_tokens=300
                        )
                    )

                    if response.text:
                        return self._parse_llm_response(response.text, 'google')

                except Exception as e:
                    continue

        except ImportError:
            # Fallback to REST API if google.generativeai not available
            try:
                headers = {"Content-Type": "application/json"}

                for model_name in self.endpoints['google']['models']:
                    try:
                        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={self.api_keys['google']}"

                        payload = {
                            "contents": [{"parts": [{"text": prompt}]}],
                            "generationConfig": {
                                "temperature": 0.3,
                                "maxOutputTokens": 300
                            }
                        }

                        response = requests.post(url, headers=headers, json=payload, timeout=30)

                        if response.status_code == 200:
                            result = response.json()
                            if 'candidates' in result and result['candidates']:
                                text = result['candidates'][0]['content']['parts'][0]['text']
                                return self._parse_llm_response(text, 'google')

                    except Exception as e:
                        continue

            except Exception as e:
                print(f"Google Gemini evaluation failed: {e}")

        return None

    def _evaluate_with_together(self, prompt: str) -> Optional[Dict]:
        """Placeholder - removed non-free service"""
        return None

    def _evaluate_with_deepinfra(self, prompt: str) -> Optional[Dict]:
        """Placeholder - removed non-free service"""
        return None

    def _parse_llm_response(self, text: str, provider: str) -> Dict:
        """Parse LLM response to extract structured evaluation"""
        try:
            score_match = re.search(r'SCORE[:\s]*(\d{1,2})', text, re.IGNORECASE)
            score = int(score_match.group(1)) if score_match else 5
            score = max(1, min(10, score))  # Clamp between 1-10

            reasoning_match = re.search(r'REASONING[:\s]*(.*?)(?=STRENGTHS|$)', text, re.IGNORECASE | re.DOTALL)
            reasoning = reasoning_match.group(1).strip() if reasoning_match else "No reasoning provided"

            strengths_match = re.search(r'STRENGTHS[:\s]*(.*?)(?=WEAKNESSES|$)', text, re.IGNORECASE | re.DOTALL)
            strengths = strengths_match.group(1).strip() if strengths_match else "No strengths identified"

            weaknesses_match = re.search(r'WEAKNESSES[:\s]*(.*?)(?=RECOMMENDATION|$)', text, re.IGNORECASE | re.DOTALL)
            weaknesses = weaknesses_match.group(1).strip() if weaknesses_match else "No weaknesses identified"

            recommendation_match = re.search(r'RECOMMENDATION[:\s]*(.*)', text, re.IGNORECASE | re.DOTALL)
            recommendation = recommendation_match.group(1).strip() if recommendation_match else "No recommendation provided"

            return {
                'score': score,
                'reasoning': reasoning,
                'strengths': strengths,
                'weaknesses': weaknesses,
                'recommendation': recommendation,
                'provider': provider,
                'raw_response': text
            }

        except Exception as e:
            print(f"Failed to parse response from {provider}: {e}")
            return {
                'score': 5,
                'reasoning': "Parse error occurred",
                'strengths': "Unable to parse",
                'weaknesses': "Unable to parse",
                'recommendation': "Unable to provide",
                'provider': provider,
                'raw_response': text
            }

    def _aggregate_llm_results(self, results: List[Dict]) -> Dict:
        """Aggregate multiple LLM evaluations into consensus score"""
        if not results:
            return self._basic_heuristic_evaluation("", "")

        scores = [r['score'] for r in results]
        avg_score = np.mean(scores)
        score_std = np.std(scores)

        # Combine feedback from all models
        all_reasoning = []
        all_strengths = []
        all_weaknesses = []
        all_recommendations = []

        for result in results:
            all_reasoning.append(f"[{result['provider']}] {result['reasoning']}")
            all_strengths.append(f"[{result['provider']}] {result['strengths']}")
            all_weaknesses.append(f"[{result['provider']}] {result['weaknesses']}")
            all_recommendations.append(f"[{result['provider']}] {result['recommendation']}")

        confidence = max(0.5, 1.0 - (score_std / 3.0))  # Higher confidence if models agree

        return {
            'score': round(avg_score, 1),
            'feedback': "\n".join(all_reasoning),
            'strengths': "\n".join(all_strengths),
            'weaknesses': "\n".join(all_weaknesses),
            'recommendations': "\n".join(all_recommendations),
            'confidence': confidence,
            'consensus_providers': [r['provider'] for r in results],
            'score_variance': score_std,
            'evaluation_method': 'multi-llm-consensus',
            'timestamp': time.time()
        }

    def detect_ai_generated_content(self, text: str) -> Dict:
        """Detect AI-generated content using available services"""
        results = []

        # GPTZero detection
        if 'gptzero' in self.api_keys:
            gptzero_result = self._detect_with_gptzero(text)
            if gptzero_result:
                results.append(gptzero_result)

        # Google-based AI detection using Gemini
        if 'google' in self.api_keys:
            google_ai_result = self._detect_with_google_ai(text)
            if google_ai_result:
                results.append(google_ai_result)

        # Enhanced pattern-based detection as baseline
        pattern_result = self._pattern_based_ai_detection(text)
        results.append(pattern_result)

        return self._aggregate_ai_detection_results(results)

    def _detect_with_google_ai(self, text: str) -> Optional[Dict]:
        """Use Google Gemini to detect AI-generated content"""
        try:
            detection_prompt = f"""Analyze the following text and determine if it appears to be AI-generated or human-written. Look for patterns like:
- Overly formal or structured language
- Generic responses without personal experience
- Perfect grammar and flow
- Repetitive phrasing patterns
- Lack of natural speech patterns

Text to analyze: "{text}"

Respond with:
AI_PROBABILITY: [number from 0.0 to 1.0]
REASONING: [brief explanation of your assessment]
INDICATORS: [specific patterns you noticed]"""

            # Try REST API approach for Google
            try:
                headers = {"Content-Type": "application/json"}
                # Changed model to gemini-1.5-flash for AI detection
                url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={self.api_keys['google']}"

                payload = {
                    "contents": [{"parts": [{"text": detection_prompt}]}],
                    "generationConfig": {
                        "temperature": 0.1,
                        "maxOutputTokens": 200
                    }
                }

                response = requests.post(url, headers=headers, json=payload, timeout=25)

                if response.status_code == 200:
                    result = response.json()
                    if 'candidates' in result and result['candidates']:
                        response_text = result['candidates'][0]['content']['parts'][0]['text']

                        # Parse the response
                        ai_prob_match = re.search(r'AI_PROBABILITY[:\s]*([\d.]+)', response_text)
                        ai_probability = float(ai_prob_match.group(1)) if ai_prob_match else 0.5

                        reasoning_match = re.search(r'REASONING[:\s]*(.*?)(?=INDICATORS|$)', response_text, re.DOTALL)
                        reasoning = reasoning_match.group(1).strip() if reasoning_match else "No reasoning provided"

                        return {
                            'ai_probability': min(1.0, max(0.0, ai_probability)),
                            'service': 'google_ai',
                            'details': {
                                'reasoning': reasoning,
                                'raw_response': response_text
                            }
                        }
            except Exception as e:
                print(f"Google AI detection REST API failed: {e}")

            # Fallback to using the SDK if available
            try:
                import google.generativeai as genai
                genai.configure(api_key=self.api_keys['google'])
                # Changed model to gemini-1.5-flash for AI detection
                model = genai.GenerativeModel('gemini-1.5-flash')

                response = model.generate_content(
                    detection_prompt,
                    generation_config=genai.types.GenerationConfig(
                        temperature=0.1,
                        max_output_tokens=200
                    )
                )

                if response.text:
                    ai_prob_match = re.search(r'AI_PROBABILITY[:\s]*([\d.]+)', response.text)
                    ai_probability = float(ai_prob_match.group(1)) if ai_prob_match else 0.5

                    return {
                        'ai_probability': min(1.0, max(0.0, ai_probability)),
                        'service': 'google_ai',
                        'details': {'reasoning': response.text}
                    }

            except ImportError:
                pass
            except Exception as e:
                print(f"Google AI detection SDK failed: {e}")

        except Exception as e:
            print(f"Google AI detection failed: {e}")
        return None

    def _detect_with_gptzero(self, text: str) -> Optional[Dict]:
        """Detect AI content using GPTZero API"""
        try:
            headers = {
                "Authorization": f"Bearer {self.api_keys['gptzero']}",
                "Content-Type": "application/json"
            }

            payload = {"document": text}

            response = requests.post(
                "https://api.gptzero.me/v2/predict/text",
                headers=headers,
                json=payload,
                timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                return {
                    'ai_probability': result.get('documents', [{}])[0].get('average_generated_prob', 0),
                    'service': 'gptzero',
                    'details': result.get('documents', [{}])[0]
                }

        except Exception as e:
            print(f"GPTZero detection failed: {e}")
        return None

    def _pattern_based_ai_detection(self, text: str) -> Dict:
        """Enhanced pattern-based AI detection"""
        ai_indicators = {
            'repetitive_phrases': 0,
            'formal_language': 0,
            'perfect_grammar': 0,
            'generic_responses': 0,
            'unnatural_flow': 0
        }

        # Check for repetitive phrases
        words = text.lower().split()
        word_counts = {}
        for word in words:
            if len(word) > 3:
                word_counts[word] = word_counts.get(word, 0) + 1

        max_repetition = max(word_counts.values()) if word_counts else 1
        if max_repetition > len(words) * 0.1:
            ai_indicators['repetitive_phrases'] = 0.3

        # Check for overly formal language
        formal_words = ['furthermore', 'consequently', 'nevertheless', 'however', 'therefore', 'moreover']
        formal_count = sum(1 for word in formal_words if word in text.lower())
        if formal_count > 2:
            ai_indicators['formal_language'] = 0.4

        # Check for perfect grammar patterns
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        if len(sentences) > 2:
            avg_sentence_length = np.mean([len(s.split()) for s in sentences])
            if avg_sentence_length > 20:  # Very long sentences
                ai_indicators['perfect_grammar'] = 0.2

        # Check for generic AI responses
        generic_phrases = [
            'as an ai', 'i don\'t have personal', 'i cannot', 'i\'m unable to',
            'based on my training', 'let me help you', 'i\'d be happy to'
        ]
        generic_count = sum(1 for phrase in generic_phrases if phrase in text.lower())
        if generic_count > 0:
            ai_indicators['generic_responses'] = 0.6

        # Calculate overall AI probability
        total_score = sum(ai_indicators.values())
        ai_probability = min(1.0, total_score)

        return {
            'ai_probability': ai_probability,
            'service': 'pattern_analysis',
            'details': ai_indicators
        }

    def _aggregate_ai_detection_results(self, results: List[Dict]) -> Dict:
        """Aggregate AI detection results from multiple services"""
        if not results:
            return {'ai_probability': 0.5, 'confidence': 0.1, 'services': []}

        probabilities = [r['ai_probability'] for r in results]
        avg_probability = np.mean(probabilities)
        prob_std = np.std(probabilities) if len(probabilities) > 1 else 0

        # Higher confidence when services agree
        confidence = max(0.3, 1.0 - prob_std)

        risk_level = 'High' if avg_probability > 0.7 else 'Medium' if avg_probability > 0.4 else 'Low'

        return {
            'ai_probability': avg_probability,
            'risk_level': risk_level,
            'confidence': confidence,
            'services': [r['service'] for r in results],
            'individual_results': results,
            'agreement_score': 1.0 - prob_std
        }

    def _basic_heuristic_evaluation(self, question: str, answer: str) -> Dict:
        """Basic fallback evaluation when all LLM services fail"""
        # Simple scoring based on length and relevance
        word_count = len(answer.split())

        if word_count < 10:
            score = 3
            feedback = "Response is too brief"
        elif word_count < 50:
            score = 5
            feedback = "Response length is adequate"
        elif word_count < 200:
            score = 7
            feedback = "Good response length with detail"
        else:
            score = 6
            feedback = "Response may be too lengthy"

        return {
            'score': score,
            'feedback': feedback,
            'strengths': "Basic evaluation performed",
            'weaknesses': "Limited analysis available",
            'recommendations': "Use API keys for detailed evaluation",
            'confidence': 0.3,
            'evaluation_method': 'basic_heuristic'
        }

class EnhancedInterviewAnalyzer:
    def __init__(self, audio_file_path):
        """
        Initialize the Enhanced Interview Analyzer with Multi-LLM support
        """
        self.audio_file_path = audio_file_path
        self.whisper_model = None
        self.segments = []
        self.speakers = {}
        self.qa_pairs = []
        self.timing_analysis = {}
        self.llm_evaluator = MultiLLMEvaluator()
        self.evaluation_results = []
        self.hiring_recommendation = {}

    def setup_api_keys(self):
        """Setup API keys for enhanced features"""
        self.llm_evaluator.collect_api_keys()

    def load_whisper_model(self, model_size="base"):
        """Load Whisper model for transcription"""
        print(f"Loading Whisper model ({model_size})...")
        self.whisper_model = whisper.load_model(model_size)
        print("✓ Whisper model loaded")

    def transcribe_audio(self):
        """Transcribe audio using Whisper with word-level timestamps"""
        print("Transcribing audio...")

        result = self.whisper_model.transcribe(
            self.audio_file_path,
            word_timestamps=True,
            verbose=False
        )

        segments = []
        for segment in result["segments"]:
            segment_data = {
                'start': segment['start'],
                'end': segment['end'],
                'text': segment['text'].strip(),
                'words': segment.get('words', [])
            }
            segments.append(segment_data)

        self.segments = segments
        print(f"✓ Transcription complete. Found {len(segments)} segments")
        return segments

    def extract_audio_features(self, segment_start, segment_end, sr=16000):
        """Extract audio features for speaker identification"""
        y, _ = librosa.load(self.audio_file_path,
                          offset=segment_start,
                          duration=segment_end - segment_start,
                          sr=sr)

        if len(y) < sr * 0.1:
            return None

        features = []

        # Pitch
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr, threshold=0.1)
        pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0
        features.append(pitch_mean)

        # MFCC features
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        features.extend(np.mean(mfcc, axis=1))

        # Spectral features
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)
        features.append(np.mean(spectral_centroids))

        # Zero crossing rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features.append(np.mean(zcr))

        # RMS energy
        rms = librosa.feature.rms(y=y)
        features.append(np.mean(rms))

        return np.array(features)

    def identify_speakers(self):
        """Identify speakers using audio feature clustering"""
        print("Identifying speakers...")

        features_list = []
        valid_segments = []

        for i, segment in enumerate(self.segments):
            features = self.extract_audio_features(segment['start'], segment['end'])
            if features is not None and not np.any(np.isnan(features)):
                features_list.append(features)
                valid_segments.append(i)

        if len(features_list) < 2:
            print("Not enough valid segments for speaker identification")
            return

        scaler = StandardScaler()
        features_normalized = scaler.fit_transform(features_list)

        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
        speaker_labels = kmeans.fit_predict(features_normalized)

        for i, segment_idx in enumerate(valid_segments):
            self.segments[segment_idx]['speaker'] = f"Speaker_{speaker_labels[i]}"

        for i, segment in enumerate(self.segments):
            if 'speaker' not in segment:
                prev_speaker = next((self.segments[j]['speaker'] for j in range(i-1, -1, -1)
                                   if 'speaker' in self.segments[j]), 'Speaker_0')
                segment['speaker'] = prev_speaker

        print("✓ Speaker identification complete")

    def determine_interviewer_interviewee(self):
        """Determine who is interviewer vs interviewee based on question patterns"""
        print("Determining interviewer and interviewee...")

        speaker_stats = {'Speaker_0': {'questions': 0, 'total_time': 0, 'segments': 0},
                        'Speaker_1': {'questions': 0, 'total_time': 0, 'segments': 0}}

        question_patterns = [
            r'\?', r'\bwhat\b', r'\bhow\b', r'\bwhy\b', r'\bwhen\b',
            r'\bwhere\b', r'\bwho\b', r'\bcan you\b', r'\bwould you\b',
            r'\bcould you\b', r'\btell me\b', r'\bexplain\b'
        ]

        for segment in self.segments:
            speaker = segment['speaker']
            text = segment['text'].lower()
            duration = segment['end'] - segment['start']

            speaker_stats[speaker]['total_time'] += duration
            speaker_stats[speaker]['segments'] += 1

            for pattern in question_patterns:
                if re.search(pattern, text):
                    speaker_stats[speaker]['questions'] += 1
                    break

        speaker_0_q_ratio = speaker_stats['Speaker_0']['questions'] / max(speaker_stats['Speaker_0']['segments'], 1)
        speaker_1_q_ratio = speaker_stats['Speaker_1']['questions'] / max(speaker_stats['Speaker_1']['segments'], 1)

        if speaker_0_q_ratio > speaker_1_q_ratio:
            interviewer = 'Speaker_0'
            interviewee = 'Speaker_1'
        else:
            interviewer = 'Speaker_1'
            interviewee = 'Speaker_0'

        self.speakers = {
            'interviewer': interviewer,
            'interviewee': interviewee
        }

        for segment in self.segments:
            if segment['speaker'] == interviewer:
                segment['role'] = 'Interviewer'
            else:
                segment['role'] = 'Interviewee'

        print(f"✓ Interviewer: {interviewer}, Interviewee: {interviewee}")
        return speaker_stats

    def extract_qa_pairs(self):
        """Extract Q&A pairs from the conversation"""
        print("Extracting Q&A pairs...")

        qa_pairs = []
        current_question = None
        current_answer_parts = []

        for i, segment in enumerate(self.segments):
            text = segment['text'].strip()
            if not text:
                continue

            if segment['role'] == 'Interviewer':
                if current_question and current_answer_parts:
                    answer_text = ' '.join([part['text'] for part in current_answer_parts])
                    qa_pairs.append({
                        'question': current_question['text'],
                        'answer': answer_text,
                        'question_start': current_question['start'],
                        'question_end': current_question['end'],
                        'answer_start': current_answer_parts[0]['start'] if current_answer_parts else 0,
                        'answer_end': current_answer_parts[-1]['end'] if current_answer_parts else 0
                    })

                current_question = {
                    'text': text,
                    'start': segment['start'],
                    'end': segment['end']
                }
                current_answer_parts = []

            elif segment['role'] == 'Interviewee' and current_question:
                current_answer_parts.append({
                    'text': text,
                    'start': segment['start'],
                    'end': segment['end']
                })

        if current_question and current_answer_parts:
            answer_text = ' '.join([part['text'] for part in current_answer_parts])
            qa_pairs.append({
                'question': current_question['text'],
                'answer': answer_text,
                'question_start': current_question['start'],
                'question_end': current_question['end'],
                'answer_start': current_answer_parts[0]['start'],
                'answer_end': current_answer_parts[-1]['end']
            })

        self.qa_pairs = qa_pairs
        print(f"✓ Extracted {len(qa_pairs)} Q&A pairs")
        return qa_pairs

    def evaluate_responses(self, position_context: str = ""):
        """Evaluate each Q&A pair using Multi-LLM consensus"""
        print("🤖 Evaluating responses with Multi-LLM consensus...")

        self.evaluation_results = []

        for i, qa in enumerate(self.qa_pairs):
            print(f"Evaluating Q&A pair {i+1}/{len(self.qa_pairs)}...")

            # Get LLM evaluation
            llm_eval = self.llm_evaluator.score_answer_with_llm(
                qa['question'],
                qa['answer'],
                position_context
            )

            # Check for AI-generated content
            ai_detection = self.llm_evaluator.detect_ai_generated_content(qa['answer'])

            # Calculate response latency
            response_latency = qa.get('answer_start', 0) - qa.get('question_end', 0)
            response_latency = max(0, response_latency)

            evaluation = {
                'qa_index': i,
                'question': qa['question'],
                'answer': qa['answer'],
                'llm_score': llm_eval['score'],
                'llm_feedback': llm_eval['feedback'],
                'llm_strengths': llm_eval.get('strengths', ''),
                'llm_weaknesses': llm_eval.get('weaknesses', ''),
                'llm_recommendations': llm_eval.get('recommendations', ''),
                'llm_confidence': llm_eval.get('confidence', 0.5),
                'consensus_providers': llm_eval.get('consensus_providers', []),
                'ai_probability': ai_detection['ai_probability'],
                'ai_risk_level': ai_detection['risk_level'],
                'ai_confidence': ai_detection['confidence'],
                'ai_detection_services': ai_detection['services'],
                'response_latency': response_latency,
                'answer_length': len(qa['answer'].split()),
                'evaluation_timestamp': time.time()
            }

            self.evaluation_results.append(evaluation)
            qa.update(evaluation)

            # Brief delay to avoid overwhelming APIs
            time.sleep(1)

        print("✓ Multi-LLM evaluation complete")
        return self.evaluation_results

    def calculate_response_latencies(self):
        """Calculate response latencies between questions and answers"""
        latencies = []
        for qa in self.qa_pairs:
            latency = qa['answer_start'] - qa['question_end']
            latencies.append(max(0, latency))
            qa['response_latency'] = max(0, latency)
        return latencies

    def analyze_timing(self):
        """Analyze speaking time distribution and response latencies"""
        print("Analyzing timing...")

        interviewer_time = sum(segment['end'] - segment['start']
                             for segment in self.segments
                             if segment['role'] == 'Interviewer')

        interviewee_time = sum(segment['end'] - segment['start']
                             for segment in self.segments
                             if segment['role'] == 'Interviewee')

        total_time = interviewer_time + interviewee_time
        latencies = self.calculate_response_latencies()

        self.timing_analysis = {
            'interviewer_time': interviewer_time,
            'interviewee_time': interviewee_time,
            'total_speaking_time': total_time,
            'interviewer_percentage': (interviewer_time / total_time) * 100,
            'interviewee_percentage': (interviewee_time / total_time) * 100,
            'response_latencies': latencies,
            'avg_response_latency': np.mean(latencies) if latencies else 0,
            'max_response_latency': np.max(latencies) if latencies else 0,
            'min_response_latency': np.min(latencies) if latencies else 0
        }

        print("✓ Timing analysis complete")
        return self.timing_analysis

    def generate_hiring_recommendation(self):
        """Generate overall hiring recommendation based on Multi-LLM analysis"""
        print("🎯 Generating hiring recommendation...")

        if not self.evaluation_results:
            print("No evaluation results available for recommendation")
            return

        # Calculate overall metrics
        avg_score = np.mean([eval_result['llm_score'] for eval_result in self.evaluation_results])
        avg_ai_probability = np.mean([eval_result['ai_probability'] for eval_result in self.evaluation_results])
        avg_response_time = np.mean([eval_result['response_latency'] for eval_result in self.evaluation_results])
        avg_confidence = np.mean([eval_result['llm_confidence'] for eval_result in self.evaluation_results])

        high_ai_risk_count = sum(1 for eval_result in self.evaluation_results
                               if eval_result['ai_risk_level'] == 'High')

        # Enhanced scoring criteria
        score_weight = 0.35
        ai_authenticity_weight = 0.25
        response_time_weight = 0.15
        consistency_weight = 0.15
        confidence_weight = 0.10

        # Calculate final score (0-100)
        score_component = (avg_score / 10) * score_weight * 100

        # AI authenticity (lower AI probability is better)
        authenticity_component = (1 - avg_ai_probability) * ai_authenticity_weight * 100

        # Response time scoring (optimal range: 1-3 seconds)
        if 1 <= avg_response_time <= 3:
            response_time_component = response_time_weight * 100
        elif avg_response_time < 1:
            response_time_component = response_time_weight * 80  # Too quick might indicate preparation
        else:
            response_time_component = max(0, response_time_weight * 100 - (avg_response_time - 3) * 10)

        # Consistency scoring
        score_std = np.std([eval_result['llm_score'] for eval_result in self.evaluation_results])
        consistency_component = max(0, consistency_weight * 100 - score_std * 10)

        # Confidence scoring
        confidence_component = avg_confidence * confidence_weight * 100

        final_score = (score_component + authenticity_component + response_time_component +
                      consistency_component + confidence_component)

        # Generate recommendation with AI considerations
        if final_score >= 85 and high_ai_risk_count == 0:
            recommendation = "STRONGLY RECOMMEND"
        elif final_score >= 70 and high_ai_risk_count <= 1:
            recommendation = "RECOMMEND"
        elif final_score >= 55 and high_ai_risk_count <= 2:
            recommendation = "CONSIDER WITH RESERVATIONS"
        elif high_ai_risk_count > len(self.evaluation_results) * 0.5:
            recommendation = "REJECT - HIGH AI ASSISTANCE DETECTED"
        else:
            recommendation = "DO NOT RECOMMEND"

        # Generate detailed reasoning
        strengths = []
        concerns = []

        if avg_score >= 7:
            strengths.append(f"Strong average response quality ({avg_score:.1f}/10)")
        elif avg_score < 5:
            concerns.append(f"Below average response quality ({avg_score:.1f}/10)")

        if avg_ai_probability < 0.3:
            strengths.append("Low AI assistance probability - responses appear authentic")
        elif high_ai_risk_count > 0:
            concerns.append(f"High AI assistance risk detected in {high_ai_risk_count} responses")

        if avg_confidence >= 0.7:
            strengths.append("High confidence in LLM evaluations indicating reliable assessment")
        elif avg_confidence < 0.4:
            concerns.append("Low confidence in evaluations - may need manual review")

        if 1 <= avg_response_time <= 3:
            strengths.append("Optimal response timing showing good preparation and thoughtfulness")
        elif avg_response_time > 5:
            concerns.append("Slow response times may indicate lack of preparation")

        if score_std < 1.5:
            strengths.append("Consistent performance across all questions")
        elif score_std > 2.5:
            concerns.append("Inconsistent performance across questions")

        # Speaking time analysis
        interviewee_percentage = self.timing_analysis.get('interviewee_percentage', 0)
        if 60 <= interviewee_percentage <= 80:
            strengths.append("Good balance of speaking time")
        elif interviewee_percentage < 50:
            concerns.append("Candidate spoke too little - may indicate lack of engagement")
        elif interviewee_percentage > 85:
            concerns.append("Candidate dominated conversation - may indicate poor listening skills")

        self.hiring_recommendation = {
            'recommendation': recommendation,
            'final_score': final_score,
            'avg_response_score': avg_score,
            'avg_ai_probability': avg_ai_probability,
            'avg_response_time': avg_response_time,
            'avg_confidence': avg_confidence,
            'high_ai_risk_count': high_ai_risk_count,
            'strengths': strengths,
            'concerns': concerns,
            'total_qa_pairs': len(self.evaluation_results),
            'interviewee_speaking_percentage': interviewee_percentage,
            'consensus_providers_used': list(set([provider for eval_result in self.evaluation_results
                                                for provider in eval_result.get('consensus_providers', [])]))
        }

        print("✓ Hiring recommendation generated")
        return self.hiring_recommendation

    def generate_report(self):
        """Generate a comprehensive analysis report"""
        print("\n" + "="*60)
        print("🎯 ENHANCED MULTI-LLM INTERVIEW ANALYSIS REPORT")
        print("="*60)

        # Basic Info
        print(f"\n📊 BASIC INFORMATION")
        print(f"Audio file: {os.path.basename(self.audio_file_path)}")
        print(f"Total segments: {len(self.segments)}")
        print(f"Q&A pairs extracted: {len(self.qa_pairs)}")

        # LLM Analysis Info
        if self.hiring_recommendation.get('consensus_providers_used'):
            print(f"LLM providers used: {', '.join(self.hiring_recommendation['consensus_providers_used'])}")

        # Speaking Time
        timing = self.timing_analysis
        print(f"\n⏱️ SPEAKING TIME ANALYSIS")
        print(f"Interviewer speaking time: {timedelta(seconds=int(timing['interviewer_time']))}")
        print(f"Interviewee speaking time: {timedelta(seconds=int(timing['interviewee_time']))}")
        print(f"Total speaking time: {timedelta(seconds=int(timing['total_speaking_time']))}")
        print(f"Interviewer percentage: {timing['interviewer_percentage']:.1f}%")
        print(f"Interviewee percentage: {timing['interviewee_percentage']:.1f}%")

        # Multi-LLM Evaluation Summary
        if self.evaluation_results:
            print(f"\n🤖 MULTI-LLM EVALUATION SUMMARY")
            avg_score = np.mean([eval_result['llm_score'] for eval_result in self.evaluation_results])
            avg_ai_prob = np.mean([eval_result['ai_probability'] for eval_result in self.evaluation_results])
            avg_confidence = np.mean([eval_result['llm_confidence'] for eval_result in self.evaluation_results])
            high_ai_count = sum(1 for eval_result in self.evaluation_results
                              if eval_result['ai_risk_level'] == 'High')

            print(f"Average LLM Score: {avg_score:.2f}/10")
            print(f"Average AI Probability: {avg_ai_prob:.2f}")
            print(f"Average Confidence: {avg_confidence:.2f}")
            print(f"High AI Risk Responses: {high_ai_count}/{len(self.evaluation_results)}")

        # Response Latency
        print(f"\n🕕 RESPONSE LATENCY ANALYSIS")
        print(f"Average response latency: {timing['avg_response_latency']:.2f} seconds")
        print(f"Minimum response latency: {timing['min_response_latency']:.2f} seconds")
        print(f"Maximum response latency: {timing['max_response_latency']:.2f} seconds")

        # Hiring Recommendation
        if self.hiring_recommendation:
            rec = self.hiring_recommendation
            print(f"\n🎯 HIRING RECOMMENDATION")
            print(f"Overall Recommendation: {rec['recommendation']}")
            print(f"Final Score: {rec['final_score']:.1f}/100")

            if rec['strengths']:
                print(f"\n✅ STRENGTHS:")
                for strength in rec['strengths']:
                    print(f"  • {strength}")

            if rec['concerns']:
                print(f"\n⚠️ CONCERNS:")
                for concern in rec['concerns']:
                    print(f"  • {concern}")

        # Detailed Q&A Analysis
        print(f"\n❓ DETAILED Q&A ANALYSIS")
        for i, eval_result in enumerate(self.evaluation_results, 1):  # Show all
            print(f"\n{i}. Q: {eval_result['question']}")
            print(f"   A: {eval_result['answer']}")
            print(f"   LLM Score: {eval_result['llm_score']:.1f}/10 (Confidence: {eval_result['llm_confidence']:.2f})")
            print(f"   AI Risk: {eval_result['ai_risk_level']} (Probability: {eval_result['ai_probability']:.2f})")
            print(f"   Response Latency: {eval_result['response_latency']:.2f}s")
            print(f"   Providers: {', '.join(eval_result.get('consensus_providers', ['N/A']))}")
            print(f"   Feedback: {eval_result['llm_feedback']}")

        if len(self.evaluation_results) > 3:
            pass # No longer truncating

    def save_results(self, output_dir="enhanced_multi_llm_analysis"):
        """Save enhanced analysis results to files"""
        os.makedirs(output_dir, exist_ok=True)

        # Save enhanced Q&A pairs with Multi-LLM evaluation
        qa_df = pd.DataFrame(self.evaluation_results)
        qa_df.to_csv(f"{output_dir}/multi_llm_qa_analysis.csv", index=False)

        # Save segments with speaker info
        segments_df = pd.DataFrame(self.segments)
        segments_df.to_csv(f"{output_dir}/segments.csv", index=False)

        # Save timing analysis
        timing_df = pd.DataFrame([self.timing_analysis])
        timing_df.to_csv(f"{output_dir}/timing_analysis.csv", index=False)

        # Save hiring recommendation
        if self.hiring_recommendation:
            recommendation_df = pd.DataFrame([self.hiring_recommendation])
            recommendation_df.to_csv(f"{output_dir}/hiring_recommendation.csv", index=False)

        # Create enhanced visualizations
        self.create_enhanced_visualizations(output_dir)

        print(f"✅ Enhanced results saved to {output_dir}/")

    def create_enhanced_visualizations(self, output_dir):
        """Create enhanced visualizations including Multi-LLM scores and AI detection"""
        fig, axes = plt.subplots(3, 2, figsize=(16, 18))

        # 1. Speaking time pie chart
        labels = ['Interviewer', 'Interviewee']
        times = [self.timing_analysis['interviewer_time'], self.timing_analysis['interviewee_time']]
        axes[0, 0].pie(times, labels=labels, autopct='%1.1f%%', startangle=90, colors=['lightblue', 'lightcoral'])
        axes[0, 0].set_title('Speaking Time Distribution')

        # 2. LLM Scores distribution
        if self.evaluation_results:
            llm_scores = [eval_result['llm_score'] for eval_result in self.evaluation_results]
            axes[0, 1].hist(llm_scores, bins=10, edgecolor='black', color='lightgreen', alpha=0.7)
            axes[0, 1].axvline(np.mean(llm_scores), color='red', linestyle='--',
                              label=f'Mean: {np.mean(llm_scores):.2f}')
            axes[0, 1].set_xlabel('LLM Score (1-10)')
            axes[0, 1].set_ylabel('Frequency')
            axes[0, 1].set_title('Multi-LLM Score Distribution')
            axes[0, 1].legend()

        # 3. Response latencies histogram
        latencies = self.timing_analysis['response_latencies']
        if latencies:
            axes[1, 0].hist(latencies, bins=15, edgecolor='black', color='orange', alpha=0.7)
            axes[1, 0].axvline(np.mean(latencies), color='red', linestyle='--',
                              label=f'Mean: {np.mean(latencies):.2f}s')
            axes[1, 0].set_xlabel('Response Latency (seconds)')
            axes[1, 0].set_ylabel('Frequency')
            axes[1, 0].set_title('Response Latency Distribution')
            axes[1, 0].legend()

        # 4. AI Detection risk analysis
        if self.evaluation_results:
            ai_probabilities = [eval_result['ai_probability'] for eval_result in self.evaluation_results]
            risk_levels = [eval_result['ai_risk_level'] for eval_result in self.evaluation_results]

            axes[1, 1].hist(ai_probabilities, bins=10, edgecolor='black', color='red', alpha=0.7)
            axes[1, 1].axvline(np.mean(ai_probabilities), color='blue', linestyle='--',
                              label=f'Mean: {np.mean(ai_probabilities):.2f}')
            axes[1, 1].set_xlabel('AI Probability')
            axes[1, 1].set_ylabel('Frequency')
            axes[1, 1].set_title('AI Detection Probability Distribution')
            axes[1, 1].legend()

        # 5. Score vs AI Probability scatter plot
        if self.evaluation_results:
            scores = [eval_result['llm_score'] for eval_result in self.evaluation_results]
            ai_probs = [eval_result['ai_probability'] for eval_result in self.evaluation_results]

            scatter = axes[2, 0].scatter(ai_probs, scores, alpha=0.6, s=60, color='purple')
            axes[2, 0].set_xlabel('AI Probability')
            axes[2, 0].set_ylabel('LLM Score (1-10)')
            axes[2, 0].set_title('LLM Score vs AI Detection')

            # Add trend line
            if len(ai_probs) > 1:
                z = np.polyfit(ai_probs, scores, 1)
                p = np.poly1d(z)
                axes[2, 0].plot(ai_probs, p(ai_probs), "r--", alpha=0.8)

        # 6. Overall performance summary
        if self.hiring_recommendation:
            rec = self.hiring_recommendation

            metrics = ['Response Quality', 'Authenticity', 'Response Speed', 'Consistency', 'Confidence']
            values = [
                rec['avg_response_score'] * 10,  # Scale to 100
                (1 - rec['avg_ai_probability']) * 100,  # Invert AI probability
                min(100, max(0, 100 - (rec['avg_response_time'] - 2) * 20)),  # Speed score
                100 - min(100, np.std([eval_result['llm_score'] for eval_result in self.evaluation_results]) * 20),  # Consistency
                rec['avg_confidence'] * 100  # Confidence
            ]

            colors = ['skyblue', 'lightgreen', 'gold', 'plum', 'lightcyan']
            bars = axes[2, 1].barh(metrics, values, color=colors, alpha=0.7)
            axes[2, 1].set_xlabel('Score (0-100)')
            axes[2, 1].set_title(f'Performance Summary\nOverall: {rec["recommendation"]}')
            axes[2, 1].set_xlim(0, 100)

            # Add value labels on bars
            for i, (bar, value) in enumerate(zip(bars, values)):
                axes[2, 1].text(value + 1, i, f'{value:.1f}', va='center')

        plt.tight_layout()
        plt.savefig(f"{output_dir}/multi_llm_analysis_visualizations.png", dpi=300, bbox_inches='tight')
        plt.show()

    def analyze_complete(self, model_size="base", save_results=True, position_context=""):
        """Run complete enhanced Multi-LLM analysis pipeline"""
        try:
            print("🚀 Starting Enhanced Multi-LLM Interview Analysis...")

            # Setup API keys first
            self.setup_api_keys()

            self.load_whisper_model(model_size)
            self.transcribe_audio()
            self.identify_speakers()
            self.determine_interviewer_interviewee()
            self.extract_qa_pairs()
            self.analyze_timing()

            # Enhanced Multi-LLM analysis
            self.evaluate_responses(position_context)
            self.generate_hiring_recommendation()

            self.generate_report()

            if save_results:
                self.save_results()

            print("\n✅ Enhanced Multi-LLM analysis completed successfully!")
            return self.hiring_recommendation

        except Exception as e:
            print(f"❌ Error during analysis: {str(e)}")
            raise

# Enhanced Usage Functions
def main():
    """Main function to run the enhanced Multi-LLM interview analyzer"""
    print("Installing required packages...")
    install_packages()

    from google.colab import files
    print("\nPlease upload your interview audio file:")
    uploaded = files.upload()

    if uploaded:
        audio_file = list(uploaded.keys())[0]
        print(f"Uploaded file: {audio_file}")

        # Get additional context
        print("\nOptional: Provide job position context for better evaluation")
        print("(e.g., 'Software Engineer', 'Marketing Manager', 'Data Scientist')")
        position = input("Position (press Enter to skip): ").strip()

        # Initialize enhanced analyzer
        analyzer = EnhancedInterviewAnalyzer(audio_file)

        # Run enhanced analysis
        recommendation = analyzer.analyze_complete(
            model_size="base",
            position_context=position
        )

        print(f"\n🎯 FINAL RECOMMENDATION: {recommendation.get('recommendation', 'Not available')}")
        print(f"📊 CONFIDENCE SCORE: {recommendation.get('final_score', 0):.1f}/100")

        return analyzer
    else:
        print("No file uploaded. Please upload an audio file to analyze.")
        return None

def analyze_interview_enhanced(audio_file_path, model_size="base", position_context=""):
    """
    Analyze an interview with enhanced Multi-LLM features

    Args:
        audio_file_path (str): Path to the audio file
        model_size (str): Whisper model size ("tiny", "base", "small", "medium", "large")
        position_context (str): Job position context for better evaluation

    Returns:
        EnhancedInterviewAnalyzer: The analyzer object with complete results
    """
    analyzer = EnhancedInterviewAnalyzer(audio_file_path)
    recommendation = analyzer.analyze_complete(
        model_size=model_size,
        position_context=position_context
    )
    return analyzer

if __name__ == "__main__":
    # Run the main function
    analyzer = main()